## Customer Support Chatbot

In [68]:


# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain_community.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain_huggingface import HuggingFaceEmbeddings

# Use local LLMs (e.g., via Ollama) for response generation
from langchain_ollama import ChatOllama

# Create prompts for the RAG system
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("✅ Libraries imported! You're good to go!")



✅ Libraries imported! You're good to go!


### 1 - Data preparation

The goal of this step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents typically come from:

    PDF files: local documents such as policies, user manuals, or guides.
    Web pages (HTML): online documentation, blog posts, or help articles.

In this step, we perform two actions:

    Ingesting: load every PDF and collect the raw text in a list named raw_docs.
    Chunking: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.

    Step 1. Load the pdfs and get the data
    Step 2. Data chunking
    Step 3. Embedding creation of chunks and indexing




## 2 - Data preparation

The goal of this step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents typically come from:

    PDF files: local documents such as policies, user manuals, or guides.
    Web pages (HTML): online documentation, blog posts, or help articles.

In this step, we perform two actions:

    Ingesting: load every PDF and collect the raw text in a list named raw_docs.
    Chunking: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.




### 2.1 - Ingest source documents

We can use different libraries to load and process PDFs. A quick web search will show several options, each with its own strengths. In this case, we’ll use PyPDFLoader from LangChain, which makes it easy to extract text from PDF files for downstream processing. To learn more about how to use it, refer to: https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/

Use PyPDFLoader to load every PDF whose filename matches data/Everstorm_*.pdf and collect all pages in a list called raw_docs. The content of these PDFs is synthetically generated for educational purposes.


In [69]:
def load_offline_files():
    pdf_paths = glob.glob("data/Everstorm_*.pdf")
    raw_docs = []

    for file_path in pdf_paths:
        try:
            loader = PyPDFLoader(file_path)
            docs = loader.load()

            # adding source to all doc in docs
            if docs:
                for doc in docs:
                    doc.metadata['source'] = file_path
                raw_docs.extend(docs)
        
        except Exception as e:
            print(f"Error loading documents {file_path} : {e}")
            continue

    print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")
    return raw_docs

### 2.1 - Load web pages

You can also pull content straight from the web. Various libraries support reading and parsing web pages directly into text, which is useful for building custom knowledge bases. One example is UnstructuredURLLoader from LangChain, which can extract readable content from raw HTML pages and return them in a structured format.

To practice, load each HTML page below and store the results in a list called raw_docs. We’ve included a few sample URLs, but you can replace them with any links you prefer.

For robustness, add an offline fallback in case a URL fails. In real projects, we typically cache fetched pages to disk, handle rate limits, and track fetch timestamps so content can be refreshed periodically without relying on live network calls during development. For this project, we don’t have offline HTML copies available, but you can still practice by loading any PDFs from the data/ folder using PyPDFLoader and appending them to raw_docs.

raw_docs is a list of Document(a class defined in LangChain) objects. 
Example: [
    Document(page_content="...", metadata={...}),
    Document(page_content="...", metadata={...}),
]

In [ ]:
def load_files(file_paths: list):
    raw_doc = []

    for path_string in file_paths:
        pdf_path = glob.glob(f"data/{path_string}*.pdf")

        if not pdf_path:
            print(f"No PDF found matching: data/{path_string}*.pdf")
            continue

        try:
            loader = PyPDFLoader(pdf_path[0])
            docs = loader.load()

            if docs:
                raw_doc.extend(docs)  # ← fixed: docs not doc

        except Exception as e:
            print(f"Error loading {pdf_path[0]}: {e}")

    return raw_doc


def load_fallback(url, url_mapping, raw_docs):
    if url not in url_mapping:
        print(f"No offline fallback configured for {url}")
        return
    offline_pdf = load_files(url_mapping[url])
    if offline_pdf:
        raw_docs.extend(offline_pdf)
    else:
        print(f"Unable to get data for {url}")


URLS = [
    "https://developer.bigcommerce.com/docs/store-operations/shipping",
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds",
]

url_mapping = {
    "https://developer.bigcommerce.com/docs/store-operations/shipping": ["Everstorm_Shipping", "Everstorm_Product"],
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds": ["Everstorm_Payment", "Everstorm_Return"],
}

raw_docs = []

for url in URLS:
    try:
        loader = UnstructuredURLLoader(urls=[url])
        docs = loader.load()

        valid_docs = [doc for doc in docs if doc.page_content.strip()]

        if valid_docs:
            for doc in valid_docs:
                doc.metadata["source"] = url
            raw_docs.extend(valid_docs)
        else:
            print(f"No useful content from {url}. Loading offline docs.")
            load_fallback(url, url_mapping, raw_docs)

    except Exception as e:
        print(f"⚠️  Web fetch failed for {url}: {e}")
        load_fallback(url, url_mapping, raw_docs)

print(f"\nTotal: {len(raw_docs)} documents loaded.")


### 2.2 - Chunk the text

Long documents won’t work well directly with most LLMs. They can easily exceed the model’s context window, making it impossible for the model to read or reason over the full text at once. Even if they fit, processing long inputs can be inefficient and lead to weaker retrieval results.

To handle this, we split large documents into smaller, overlapping chunks. Several libraries can help with text splitting, each designed to preserve structure or balance chunk size. A popular choice is RecursiveCharacterTextSplitter from LangChain, which splits text intelligently while keeping paragraph or sentence boundaries intact. To familiarize youself with the library, visit: https://python.langchain.com/api_reference/text_splitters/character/langchain_text_splitters.character.RecursiveCharacterTextSplitter.html

In this project, we’ll split each document into chunks of roughly 300 tokens with a 30-token overlap using RecursiveCharacterTextSplitter. This overlap helps maintain continuity across chunks while ensuring each piece stays small enough for embedding and retrieval.



In [70]:
raw_docs = load_offline_files()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = text_splitter.split_documents(documents=raw_docs)
print(f"{len(chunks)} chunks ready for embedding")

Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)


Loaded 8 PDF pages from 4 files.
42 chunks ready for embedding


## 3 - Build a Retriever

A retriever lets the RAG pipeline efficiently look up small, relevant pieces of context at query-time. This step has two-parts:

1. **Load a model to generate embeddings:** convert each text chunk from the reference documents into a fixed-length vector that captures its semantic meaning
2. **Build vector database:** store these embeddings in a vector database

### 3.1 - Load a model to generate embeddings


The goal of this step is to convert each document chunk into a numerical vector (an embedding) that captures its semantic meaning. These embeddings allow our retriever to find and compare similar pieces of text efficiently.

There are models trained specifically for this purpose, called embedding models. One popular example is OpenAI’s `text-embedding-3-small`, which produces high-quality embeddings that work well for retrieval and semantic search.

If you prefer running everything locally, you can use smaller open-source models such as `gte-small` (77 M parameters). These local models load quickly, don’t require internet access, and are ideal for experimentation or environments without API access. However, they’re typically less powerful than hosted models.

Alternatively, you can connect to an API service to access stronger models like OpenAI’s. These require setting an API key (for example, OPENAI_API_KEY) in your environment. OpenAI allows you to create a free account and sometimes offers limited trial credits for new users, but ongoing access requires a billing setup.

In this exercise, we’ll stick to the smaller gte-small model for simplicity and reproducibility. We'll use our imported `SentenceTransformerEmbeddings` library to load the model and use it to embed queries. To learn more about lagnchain's embedding support, visit: https://python.langchain.com/docs/integrations/text_embedding/


In [71]:
emb_model = HuggingFaceEmbeddings(model_name='thenlper/gte-small')

# trying the embedding vector for a simple sentence
embedding_vector = emb_model.embed_query("Hello World")
print(len(embedding_vector))

384


### 3.2 - Build a vector database
Once we have embeddings, we need a way to store and search them efficiently. A simple list wouldn't scale well, especially when we have thousands of chunks and need to quickly find the most relevant ones.

To solve this, we use FAISS, an open-source similarity search library developed by Meta. FAISS is optimized for fast nearest-neighbour search in high-dimensional spaces, making it ideal for tasks like semantic retrieval and recommendation. 
It's strongly encouraged to visit their quickstart guide to understand how FAISS works and how to use it effectively: https://github.com/facebookresearch/faiss/wiki/getting-started

In this step, we'll feed all our document embeddings into FAISS, which builds an in-memory vector index. This index allows us to efficiently query for the k most similar chunks to any given question.

During inference, we'll use this index to retrieve the top-k relevant chunks and pass them to the LLM as context, enabling it to answer questions grounded in our document.

In [72]:
# Expected steps:
    # 1. Build the FAISS index from the list of document chunks and their embeddings.
    # 2. Create a retriever object with a suitable k value (e.g., 8).
    # 3. Save the vector store locally (e.g., under "faiss_index").
    # 4. Print a short confirmation showing how many embeddings were stored.

# Till now, I just have an embedding model in my code. 
# I need to create vectors of the chunks I have, map original data with the id of the vectors and store the data somewhere

vectordb = FAISS.from_documents(documents=chunks, embedding=emb_model)
# Now the function from_documents will handle creating embeddings for all the chunks, store these embeddings along with their ids in RAM
vectordb.save_local("faiss_index")



## 4 - Build the generation engine

At the core of any RAG system lies an LLM. The retriever finds relevant information, and the LLM uses that information to generate coherent, context-aware responses.

In this project, we'll use `Gemma 3 (1B)`, a small but capable open-weight model, and run it entirely on your local machine using Ollama. This means you won't need API keys or internet access to generate responses once the model is downloaded.



**What is Ollama?**
Ollama is a light weight runtime for managing and serving open-weight LLMs locally. It provides:
- A simple REST API running at localhost:11434, so your code can interact with the model via standard HTTP calls.
- A model registry and comand-line tool to pull, run, and manage models easily.
- Support for a wide variety of models (e.g., Gemma, Llama, Mistral, Phi, etc.), making it ideal for experimentation.

To learn more about Ollama, visit: https://github.com/ollama/ollama. You can browse all supported models and their sizes here: https://ollama.com/library



### 4.1 - Install `ollama` and serve `gemma3`

Follow these steps to set up Ollama and start the model server:

**1 - Install**
```bash
# macOS (Homebrew)
brew install ollama
# Linux
curl -fsSL https://ollama.com/install.sh | sh
```

If you’re on Windows, install using the official installer from https://ollama.com/download.

**2 - Start the Ollama server (keep this terminal open)**
```bash
ollama serve
```
This command launches a local server at http://localhost:11434, which will stay running in the background.


**3 - Pull the Gemma mode (or the model of your choice) in a new terminal**
```bash
ollama pull gemma3:1b
```

This downloads the 1B version of Gemma 3, a compact model suitable for running on most modern laptops. Once downloaded, Ollama will automatically handle model loading and caching.


After this setup, your system is ready to generate responses locally using the Gemma model through the Ollama API.



In [73]:
llm = ChatOllama(model="gemma3:1b", temperature=0.1)
print(llm.invoke("Tell me a fun fact about raspberry"))

content="Okay, here's a fun fact about raspberries:\n\n**Raspberries are actually a type of berry, but they're incredibly difficult to pollinate!** They require a specific type of bee, the *Melittus* species, to transfer pollen from one raspberry to another. It's a surprisingly complex pollination process! \n\nPretty cool, right? 😊 \n\nWould you like to know another fun fact about raspberries?" additional_kwargs={} response_metadata={'model': 'gemma3:1b', 'created_at': '2026-02-11T17:31:16.357607Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1895579792, 'load_duration': 260170667, 'prompt_eval_count': 16, 'prompt_eval_duration': 54624375, 'eval_count': 89, 'eval_duration': 1477255921, 'logprobs': None, 'model_name': 'gemma3:1b'} id='run--019c4dc1-d49c-7f60-b2d8-45a3b36cae60-0' usage_metadata={'input_tokens': 16, 'output_tokens': 89, 'total_tokens': 105}


## Build a RAG

### 5.1 - Define system prompt

At this stage, we need to tell the model how to behave when generating answers. The **system prompt** acts as the model's rulebook. It should clearly instruct the model to answer only using the retrieved context and to admit when it doesn't know the answer. This helps prevent hallucination and keeps the responses grounded in the provided documents.

In general, a good RAG prompt emphasizes three things: stay within context, stay factual, and stay concise. This is important because RAG works by grounding the LLM in retrieved text. If the prompt is vague, the model may invent details. A precise system prompt reduces hallucinations and keeps answers aligned with your corpus.

In [74]:
SYSTEM_PROMPT = """
You are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with "I'm not sure from the docs."

Rules:
1) Answer ONLY using the CONTEXT below. Nothing else.
2) If the CONTEXT does not contain the answer, response EXACTLY with:
    "I don't know based on the retrieved documents."
3) Do NOT use your own knowledge. Do NOT make up answers.
4) Do NOT refuse questions for privacy or safety reasons.
    Simple check the CONTEXT and respond accordingly.

CONTEXT:
{context}
"""

## 5.2 - Create a RAG chain

Now that we have a retriever, a prompt, and a language model, we can connect them into a single RAG pipeline. The retriever finds the most relevant chunks from our vector index, the prompt injects those chunks into the system message, and the LLM uses that context to produce the final answer. (retriever -> prompt -> model)


This connection handled through LangChain's `ConversationalRetrievalChain`, which combines retrieval and generation. To familiarize yourself with the library, visit: 

In [76]:
# Here we are asking the vectordb to act as a retriever which will retrieve 8 matching docs. 
# Whenever we get a query, we need to first create an embedding of the query. 
# Then, we need to perform FAISS index search for vectors. The search will return numerical ids of matching vectors
# Then we need to find the corresponding doc from the index_to_docstore_id mapping
# Retrive the document from doc store 
# Return the document.
# All these steps will be handled in the invoke method which we will call on the retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 8})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

class ManualRAG:
    def __init__(self):
        self.chat_history = []
    
    def rewrite_query(self, question: str) -> str:
        """Turn follow-up questions into standalone queries."""
        if not self.chat_history:
            return question

        messages = [
            SystemMessage(content=(
                "Given the chat history and a follow-up question, "
                "rewrite it as a standalone question. "
                "Return ONLY the rewritten question"
            )),
            *self.chat_history,
            HumanMessage(content=question)
        ]

        # *self.chat_history creates a flat list

        rewritten_question = llm.invoke(messages)
        print(rewritten_question)
        return rewritten_question.content
    
    def chat(self, question: str) -> dict:
        # Step 1: Rewrite follow-ups into standalone question
        standalone_query = self.rewrite_query(question=question)

        # Step 2: Retrieve relevant docs
        docs = retriever.invoke(standalone_query)
        context = format_docs(docs)

        # Step 3: Build message list with history and generate
        messages = [
            SystemMessage(content=SYSTEM_PROMPT.format(context=context)),
            *self.chat_history,
            HumanMessage(content=question)
        ]

        response = llm.invoke(messages)

        # Step 4: Save this turn to history
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=response.content))

        print(response.content)
        return {
            "answer" : response.content,
            "source_documents": docs
        }
    
    def reset(self):
        self.chat_history.clear()


In [79]:
# if you want to use LangGraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

#retriever = vectordb.as_retriever(search_kwargs={"k": 8})
# Shared State
class RAGState(TypedDict):
    question: str
    chat_history: list
    rewritten_query: str
    context: str
    source_documents: list
    answer: str

# Defining Nodes

def rewrite_query_node(state: RAGState) -> dict:
    """Rewrites follow-up questions into standalone queries."""
    if not state["chat_history"]:
        return {"rewritten_query": state["question"]}
    
    messages = [
        SystemMessage(content=(
            "Given the chat history and a follow-up question, "
            "rewrite it as a standalone question. "
            "Return ONLY the rewritten question."
        )),
        *state["chat_history"],
        HumanMessage(content=state["question"])
    ]

    rewritten_question = llm.invoke(messages)
    
    return {"rewritten_query": rewritten_question.content}


def retrieve_node(state: RAGState) -> dict:
    """Fetches relevant docs from FAISS using the rewritten query"""
    docs = retriever.invoke(state["rewritten_query"])
    return {
        "context": format_docs(docs),
        "source_documents": docs
    }

def generate_nodes(state: RAGState) -> dict:
    """Generates the final answer using context + chat history."""
    messages = [
        SystemMessage(content=SYSTEM_PROMPT.format(context=state["context"])),
        *state["chat_history"],
        HumanMessage(content=state["question"])
    ]
    response = llm.invoke(messages)

    updated_history = state["chat_history"] + [
        HumanMessage(content=state["question"]),
        AIMessage(content=response.content)
    ]

    return {
        "answer" : response.content,
        "chat_history" : updated_history
    }


# Building graph
graph = StateGraph(RAGState)

graph.add_node("rewrite", rewrite_query_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_nodes)

graph.add_edge(START, "rewrite")
graph.add_edge("rewrite", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

rag_chain = graph.compile()


class LangGraphRAG:
    def __init__(self):
        self.chat_history: list = []

    def chat(self, question: str) -> dict:
        result = rag_chain.invoke({
            "question": question,
            "chat_history": self.chat_history,
            "rewritten_query": "",
            "context": "",
            "source_documents": [],
            "answer": ""
         })
        
        self.chat_history = result["chat_history"]

        return {
            "answer": result["answer"],
            "source_documents": result["source_documents"]
        }
    
    def reset(self):
        self.chat_history.clear()


test_questions = [
        "What is your refund policy?",
        "How long do I have?",           # follow-up
        "What about international orders?",  # another follow-up
    ]

bot = LangGraphRAG()
for q in test_questions:
    result = bot.chat(q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer'][:300]}")
        




Q: What is your refund policy?
A: I'm not sure from the docs.

Q: How long do I have?
A: I'm not sure from the docs.

Q: What about international orders?
A: I'm not sure from the docs.


In [78]:
test_questions = [
    "If I'm not happy with my purchase, what is your refund policy and how do I start a return?",
    "How long will delivery take for a standard order, and where can I track my package once it ships?",
    "What's the quickest way to contact your support team, and what are your operating hours?",
    "Can you give me email for support team"
]

bot = ManualRAG()
for q in test_questions:
    result = bot.chat(q)
    print(f"\nQ: {q}\nA: {result['answer']}...")

We issue refunds the same day your return is scanned. Banks typically post credit within:

*   Card: 5-7 banking days
*   PayPal: Immediate
*   Klarna: 2-5 days

We also have a policy for unauthorized or heavily worn items:

*   Items showing heavy wear will be shipped back or donated to charity at our discretion, minus a $15 handling fee.

To start a return, you’ll need to:

1.  **Contact us:**  Go to returns@everstorm.example and send us a photo of the item and the reason for the return.
2.  **Print the prepaid label:** We’ll email you a prepaid label to print.
3.  **Pack securely:** Pack the item carefully to prevent damage during shipping.
4.  **Ship the item:** Send it back to us with the prepaid label.

Do you have any specific questions about the process?

Q: If I'm not happy with my purchase, what is your refund policy and how do I start a return?
A: We issue refunds the same day your return is scanned. Banks typically post credit within:

*   Card: 5-7 banking days
*   PayPal:

## 6 - Build the Streamlit UI

The goal here is to create a tiny demo so you can interact with your RAG system. 

There are many ways to create a UI. Some frameworks are powerful but takes longer to set up, while others are simple and good for quick experiments. Streamlit is a common choice for fast prototyping because it lets you make a usable interface with only a few lines of Python. If you want to learn the basics, see the Streamlit Quickstart: https://docs.streamlit.io/deploy/streamlit-community-cloud/get-started/quickstart


In this cell, we write a minimal **`app.py`** that starts a simple chat UI and calls your RAG chain.

In [ ]:
%%bash
cat > app.py << 'PY'
import pathlib, streamlit as st
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import Ollama


st.set_page_config(page_title="Customer Support Chatbot")
st.title("Customer Support Chatbot")

@st.cache_resource
def init_chain():
       vectordb = FAISS.load_local(
        "faiss_index",
        
       )


